![image info](https://raw.githubusercontent.com/albahnsen/MIAD_ML_and_NLP/main/images/banner_1.png)

# Taller: Construcción e implementación de modelos Bagging, Random Forest y XGBoost

En este taller podrán poner en práctica sus conocimientos sobre la construcción e implementación de modelos de Bagging, Random Forest y XGBoost. El taller está constituido por 8 puntos, en los cuales deberan seguir las intrucciones de cada numeral para su desarrollo.

## Datos predicción precio de automóviles

En este taller se usará el conjunto de datos de Car Listings de Kaggle donde cada observación representa el precio de un automóvil teniendo en cuenta distintas variables como año, marca, modelo, entre otras. El objetivo es predecir el precio del automóvil. Para más detalles puede visitar el siguiente enlace: [datos](https://www.kaggle.com/jpayne/852k-used-car-listings).

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Importación de librerías
%matplotlib inline
import pandas as pd

# Lectura de la información de archivo .csv
data = pd.read_csv('https://raw.githubusercontent.com/albahnsen/MIAD_ML_and_NLP/main/datasets/dataTrain_carListings.zip')

# Preprocesamiento de datos para el taller
data = data.loc[data['Model'].str.contains('Camry')].drop(['Make', 'State'], axis=1)
data = data.join(pd.get_dummies(data['Model'], prefix='M'))
data = data.drop(['Model'], axis=1)

# Visualización dataset
data.head()

,Price,Year,Mileage,M_Camry,M_Camry4dr,M_CamryBase,M_CamryL,M_CamryLE,M_CamrySE,M_CamryXLE
7,21995,2014,6480,False,False,False,True,False,False,False
11,13995,2014,39972,False,False,False,False,True,False,False
167,17941,2016,18989,False,False,False,False,False,True,False
225,12493,2014,51330,False,False,False,True,False,False,False
270,7994,2007,116065,False,True,False,False,False,False,False


In [3]:
# Separación de variables predictoras (X) y variable de interés (y)
y = data['Price']
X = data.drop(['Price'], axis=1)
# Transformo el tipo de datos en reales para las columnas tipo bool
X = X.astype({col: int for col in X.select_dtypes(include='bool').columns})

In [4]:
# Separación de datos en set de entrenamiento y test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

### Punto 1 - Árbol de decisión manual

En la celda 1 creen un árbol de decisión **manualmente**  que considere los set de entrenamiento y test definidos anteriormente y presenten el RMSE y MAE del modelo en el set de test.

In [5]:
# Celda 1
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
#Arbol de decisión manual
class Node():
    def __init__(self, feature_index=None, threshold=None, left=None, right=None, var_red=None, value=None):

        # for decision node
        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right
        self.var_red = var_red

        # for leaf node
        self.value = value

class DecisionTreeRegressor():
    def __init__(self, min_samples_split=2, max_depth=2):
        ''' constructor '''

        # initialize the root of the tree
        self.root = None

        # stopping conditions
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth

    def build_tree(self, dataset, curr_depth=0):

        X, Y = dataset[:,:-1], dataset[:,-1]
        num_samples, num_features = np.shape(X)
        best_split = {}
        # split until stopping conditions are met
        if num_samples>=self.min_samples_split and curr_depth<=self.max_depth:
            # find the best split
            best_split = self.get_best_split(dataset, num_samples, num_features)
            # check if information gain is positive
            if best_split["var_red"]>0:
                # recur left
                left_subtree = self.build_tree(best_split["dataset_left"], curr_depth+1)
                # recur right
                right_subtree = self.build_tree(best_split["dataset_right"], curr_depth+1)
                # return decision node
                return Node(best_split["feature_index"], best_split["threshold"],
                            left_subtree, right_subtree, best_split["var_red"])

        # compute leaf node
        leaf_value = self.calculate_leaf_value(Y)
        # return leaf node
        return Node(value=leaf_value)

    def get_best_split(self, dataset, num_samples, num_features):

        # dictionary to store the best split
        best_split = {}
        max_var_red = -float("inf")
        # loop over all the features
        for feature_index in range(num_features):
            feature_values = dataset[:, feature_index]
            possible_thresholds = np.unique(feature_values)
            # loop over all the feature values present in the data
            for threshold in possible_thresholds:
                # get current split
                dataset_left, dataset_right = self.split(dataset, feature_index, threshold)
                # check if childs are not null
                if len(dataset_left)>0 and len(dataset_right)>0:
                    y, left_y, right_y = dataset[:, -1], dataset_left[:, -1], dataset_right[:, -1]
                    # compute information gain
                    curr_var_red = self.variance_reduction(y, left_y, right_y)
                    # update the best split if needed
                    if curr_var_red>max_var_red:
                        best_split["feature_index"] = feature_index
                        best_split["threshold"] = threshold
                        best_split["dataset_left"] = dataset_left
                        best_split["dataset_right"] = dataset_right
                        best_split["var_red"] = curr_var_red
                        max_var_red = curr_var_red

        # return best split
        return best_split

    def split(self, dataset, feature_index, threshold):

        dataset_left = np.array([row for row in dataset if row[feature_index]<=threshold])
        dataset_right = np.array([row for row in dataset if row[feature_index]>threshold])
        return dataset_left, dataset_right

    def variance_reduction(self, parent, l_child, r_child):

        weight_l = len(l_child) / len(parent)
        weight_r = len(r_child) / len(parent)
        reduction = np.var(parent) - (weight_l * np.var(l_child) + weight_r * np.var(r_child))
        return reduction

    def calculate_leaf_value(self, Y):

        val = np.mean(Y)
        return val

    def print_tree(self, tree=None, indent=" "):

        if not tree:
            tree = self.root

        if tree.value is not None:
            print(tree.value)

        else:
            print("X_"+str(tree.feature_index), "<=", tree.threshold, "?", tree.var_red)
            print("%sleft:" % (indent), end="")
            self.print_tree(tree.left, indent + indent)
            print("%sright:" % (indent), end="")
            self.print_tree(tree.right, indent + indent)

    def fit(self, X, Y):

        dataset = np.concatenate((X, Y), axis=1)
        self.root = self.build_tree(dataset)

    def make_prediction(self, x, tree):

        if tree.value!=None: return tree.value
        feature_val = x[tree.feature_index]
        if feature_val<=tree.threshold:
            return self.make_prediction(x, tree.left)
        else:
            return self.make_prediction(x, tree.right)

    def predict(self, X):

        preditions = [self.make_prediction(x, self.root) for x in X]
        return preditions

In [8]:
import numpy as np

class Node:
    def __init__(self, feature_index=None, threshold=None, left=None, right=None, var_red=None, value=None):
        self.feature_index = feature_index  # Índice de la característica para dividir
        self.threshold = threshold          # Umbral de división
        self.left = left                    # Subárbol izquierdo
        self.right = right                  # Subárbol derecho
        self.var_red = var_red              # Reducción de varianza de la división
        self.value = value                  # Valor de la hoja (media)

class DecisionTreeRegressor:
    def __init__(self, min_samples_split=2, max_depth=2):
        self.root = None
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth

    def fit(self, X, Y):
        dataset = np.concatenate((X, Y), axis=1)
        self.root = self._build_tree(dataset)

    def _build_tree(self, dataset, depth=0):
        X, Y = dataset[:, :-1], dataset[:, -1]
        n_samples, n_features = X.shape

        if n_samples >= self.min_samples_split and depth <= self.max_depth:
            best_split = self._get_best_split(dataset, n_features)
            if best_split and best_split['var_red'] > 0:
                left = self._build_tree(best_split['left'], depth + 1)
                right = self._build_tree(best_split['right'], depth + 1)
                return Node(best_split['feature'], best_split['threshold'], left, right, best_split['var_red'])

        return Node(value=np.mean(Y))

    def _get_best_split(self, dataset, n_features):
        best = {}
        max_var_red = -np.inf
        for feature in range(n_features):
            thresholds = np.unique(dataset[:, feature])
            for threshold in thresholds:
                left, right = self._split(dataset, feature, threshold)
                if len(left) and len(right):
                    y, y_left, y_right = dataset[:, -1], left[:, -1], right[:, -1]
                    var_red = self._variance_reduction(y, y_left, y_right)
                    if var_red > max_var_red:
                        best = {
                            'feature': feature,
                            'threshold': threshold,
                            'left': left,
                            'right': right,
                            'var_red': var_red
                        }
                        max_var_red = var_red
        return best

    def _split(self, dataset, feature, threshold):
        left = dataset[dataset[:, feature] <= threshold]
        right = dataset[dataset[:, feature] > threshold]
        return left, right

    def _variance_reduction(self, parent, left, right):
        w_left, w_right = len(left) / len(parent), len(right) / len(parent)
        return np.var(parent) - (w_left * np.var(left) + w_right * np.var(right))

    def _predict(self, x, node):
        if node.value is not None:
            return node.value
        if x[node.feature_index] <= node.threshold:
            return self._predict(x, node.left)
        else:
            return self._predict(x, node.right)

    def predict(self, X):
        return [self._predict(x, self.root) for x in X]

    def print_tree(self, node=None, indent="  "):
        node = node or self.root
        if node.value is not None:
            print(f"{indent}Valor hoja: {node.value:.4f}")
        else:
            print(f"{indent}X[{node.feature_index}] <= {node.threshold:.4f}  (VarRed: {node.var_red:.4f})")
            print(f"{indent}├─ Izquierda:")
            self.print_tree(node.left, indent + "  ")
            print(f"{indent}└─ Derecha:")
            self.print_tree(node.right, indent + "  ")

In [9]:
# Entreno el modelo

y_train_s = y_train.values.reshape(-1,1)
regressor = DecisionTreeRegressor(min_samples_split=10, max_depth=9)
regressor.fit(X_train, y_train_s)

y_pred = regressor.predict(X_test.to_numpy())
np.sqrt(mean_squared_error(y_test, y_pred))

# Calcular RMSE y MAE
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"\nRMSE en el set de test: {rmse:.4f}")
print(f"MAE en el set de test: {mae:.4f}")



RMSE en el set de test: 1687.7394
MAE en el set de test: 1219.1276


### Punto 2 - Bagging manual

En la celda 2 creen un modelo bagging **manualmente** con 10 árboles de regresión y comenten sobre el desempeño del modelo.

In [7]:
# Celda 2
from sklearn.tree import DecisionTreeRegressor

# Semilla para reproducibilidad
np.random.seed(123)
n_samples = X.shape[0]
n_B = 10

# Creamos 10 muestras bootstrap (índices)
samples = [np.random.choice(a=n_samples, size=n_samples, replace=True) for _ in range(n_B)]

# DataFrame para guardar predicciones
y_pred = pd.DataFrame(index=X_test.index, columns=range(n_B))

# Entrenamos un árbol por muestra
for i, sample in enumerate(samples):
    X_b = X.iloc[sample, :]
    y_b = y.iloc[sample]
    model = DecisionTreeRegressor(max_depth=None, random_state=123 + i)
    model.fit(X_b, y_b)
    y_pred.iloc[:, i] = model.predict(X_test)

# Evaluación individual
for i in range(n_B):
    error = np.sqrt(mean_squared_error(y_test, y_pred.iloc[:, i]))
    print(f'Árbol {i} - RMSE: {error:.4f}')

# Predicción promedio del ensemble
y_pred_mean = y_pred.mean(axis=1)
ensemble_rmse = np.sqrt(mean_squared_error(y_test, y_pred_mean))
print(f'\nBagging (promedio de {n_B} árboles) - RMSE: {ensemble_rmse:.4f}')

Árbol 0 - RMSE: 1360.3012
Árbol 1 - RMSE: 1297.9989
Árbol 2 - RMSE: 1297.5757
Árbol 3 - RMSE: 1303.6690
Árbol 4 - RMSE: 1293.5805
Árbol 5 - RMSE: 1293.8224
Árbol 6 - RMSE: 1302.9088
Árbol 7 - RMSE: 1343.4339
Árbol 8 - RMSE: 1318.0425
Árbol 9 - RMSE: 1337.1796

Bagging (promedio de 10 árboles) - RMSE: 768.2936


El desempeño del modelo con bagging es significativamente mejor, pues el RMSE de los arboles individuales va desde 1297.5757 hasta 1360.3012, pero al utilizar el modelo con bagging se obtiene un RMSE de 768.2936, es decir que se reduce el RMSE del arbol 0 casi a la mitad.

### Punto 3 - Bagging con librería

En la celda 3, con la librería sklearn, entrenen un modelo bagging con 10 árboles de regresión y el parámetro `max_features` igual a `log(n_features)` y comenten sobre el desempeño del modelo.

In [ ]:
# Celda 3


### Punto 4 - Random forest con librería

En la celda 4, usando la librería sklearn entrenen un modelo de Randon Forest para regresión  y comenten sobre el desempeño del modelo.

In [ ]:
# Celda 4


### Punto 5 - Calibración de parámetros Random forest

En la celda 5, calibren los parámetros max_depth, max_features y n_estimators del modelo de Randon Forest para regresión, comenten sobre el desempeño del modelo y describan cómo cada parámetro afecta el desempeño del modelo.

In [ ]:
# Celda 5


### Punto 6 - XGBoost con librería

En la celda 6 implementen un modelo XGBoost de regresión con la librería sklearn y comenten sobre el desempeño del modelo.

In [ ]:
# Celda 6


### Punto 7 - Calibración de parámetros XGBoost

En la celda 7 calibren los parámetros learning rate, gamma y colsample_bytree del modelo XGBoost para regresión, comenten sobre el desempeño del modelo y describan cómo cada parámetro afecta el desempeño del modelo.

In [ ]:
# Celda 7


### Punto 8 - Comparación y análisis de resultados
En la celda 8 comparen los resultados obtenidos de los diferentes modelos (random forest y XGBoost) y comenten las ventajas del mejor modelo y las desventajas del modelo con el menor desempeño.

In [ ]:
# Celda 8
